# Seamless tiling, batch: stock vs periodic RoPE + seamless VAE

For every prompt of an upsampled prompts file (from `upsample_prompts.ipynb`) this generates, with the same seed and noise:

- **stock**: original `flux3.dit.rope_embed`, stock VAE decoder
- **seamless**: all-round quantized RoPE on h and w, VAE decoder with wrap-around padding (`seamless.py`)

and saves, under `OUT_DIR`: `<name>_stock.png`, `<name>_seamless.png`, their 2x2 tilings, and `scores.json` with the
seam scores (wrap-around edge difference over interior difference, 1.0 = invisible seam). `MODEL` picks `distilled`
(16 steps, no CFG) or `base` (50 steps, CFG 4). Already generated names are skipped on rerun.

## Pick a GPU

Nothing runs until `GPU` is set. The index is the one `nvidia-smi` prints; it is exported as `CUDA_VISIBLE_DEVICES`
*before* torch is imported, so inside the notebook the chosen card is always `cuda:0` (vLLM has no device argument and
only honours this variable). Restart the kernel to switch.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

GPU = None  # <-- set me, e.g. GPU = 0

print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
assert GPU is not None, "set GPU to one of the indices above before running anything else"
assert "torch" not in sys.modules, "torch is already imported: restart the kernel, then run this cell first"
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU)
DEVICE = "cuda"  # == cuda:0 == physical GPU above

REPO = Path.cwd().resolve()
REPO = REPO.parent if REPO.name == "notebooks" else REPO
os.environ.setdefault("WEIGHTS_DIR", str(REPO / "weights"))  # flux3 reads this at import time
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))


## Settings

In [ ]:
import json

import flux3.dit as dit_module
import matplotlib.pyplot as plt
import torch
from PIL import Image

from flux3.autoencoder import latent_shape
from flux3.pipeline import MODELS, WEIGHTS, Pipeline, dit_path
from seamless import decode, quantized_rope_embed, read_prompts, sample, seam_score, tiled

MODEL = "distilled"  # or "base"
PROMPTS_IN = REPO / "notebooks" / "outputs" / "seamless_prompts_upsampled.txt"
OUT_DIR = REPO / "notebooks" / "outputs" / f"seamless_batch_{MODEL}"
WIDTH, HEIGHT = 1024, 1024
SEED = 1
TILE = ("h", "w")  # axes made periodic, for the RoPE and the VAE alike
RULE = "round"

prompts = read_prompts(PROMPTS_IN)
print(f"{len(prompts)} prompts from {PROMPTS_IN}\nresults -> {OUT_DIR}")

## Models

In [ ]:
entry = WEIGHTS / MODELS[MODEL]["dit"]
pipe = Pipeline({MODEL: str(dit_path(MODEL))}, str(entry / "text_encoder"), str(entry / "ae.safetensors"), device=DEVICE)
dit = pipe.dits[MODEL]
for module in (dit, pipe.autoencoder, pipe.text_encoder.model):
    module.eval().requires_grad_(False)

h_lat, w_lat = latent_shape(HEIGHT, WIDTH)
methods = {
    "stock": (dit_module.rope_embed, ()),
    "seamless": (quantized_rope_embed(h_lat, w_lat, tile=TILE, rule=RULE), TILE),
}
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

## Generate

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
scores_path = OUT_DIR / "scores.json"
scores = json.loads(scores_path.read_text()) if scores_path.exists() else {}

for name, prompt in prompts.items():
    if name in scores:
        print(f"[{name}] already done, skipping")
        continue
    scores[name] = {}
    for method, (rope_embed, tile) in methods.items():
        tokens, ids = sample(pipe, dit, MODEL, rope_embed, prompt, WIDTH, HEIGHT, seed=SEED)
        image = decode(pipe, tokens, ids, tile)
        image.save(OUT_DIR / f"{name}_{method}.png")
        tiled(image).save(OUT_DIR / f"{name}_{method}_tiled.png")
        scores[name][method] = seam_score(image)
    scores_path.write_text(json.dumps(scores, indent=1) + "\n")
    s = scores[name]
    print(f"[{name}]  seam w: {s['stock']['w']:.2f} -> {s['seamless']['w']:.2f}   h: {s['stock']['h']:.2f} -> {s['seamless']['h']:.2f}")

## Overview

Tiled 2x2, stock on the left, seamless on the right.

In [ ]:
names = [n for n in prompts if n in scores]
fig, axes = plt.subplots(len(names), 2, figsize=(14, 7 * len(names)))
for row, name in enumerate(names):
    for col, method in enumerate(methods):
        axes[row, col].imshow(Image.open(OUT_DIR / f"{name}_{method}_tiled.png"))
        s = scores[name][method]
        axes[row, col].set_title(f"{name} / {method}   seam w {s['w']:.2f}  h {s['h']:.2f}")
        axes[row, col].axis("off")
plt.tight_layout()
fig.savefig(OUT_DIR / "overview.jpg", dpi=60)
plt.show()